# TechArena CTR: numeric neural event-choice model
Both private causal feature/history inputs required. Eight frozen epochs, fitting-only normalization, symmetric mean/max candidate pooling, explicit no-click option and fixed 10% rank blend.
Day6 gain gate then unchanged day5 confirmation. Local reused development cohorts; 0.83 is unproven.


In [ ]:
from pathlib import Path
import hashlib, importlib.util
import torch
assert torch.cuda.is_available(), "Enable Kaggle GPU"
roots=list(Path("/kaggle/input").rglob("bundle_manifest.json"))
neural_roots=list(Path("/kaggle/input").rglob("neural_manifest.json"))
assert len(roots)==len(neural_roots)==1
DATA_ROOT=roots[0].parent
NEURAL_ROOT=neural_roots[0].parent
weighting_path=Path("/kaggle/working/tabular_choice_weighting.py")
weighting_path.write_text("\"\"\"Fixed-capacity CatBoost GPU expert on the existing causal feature bundle.\r\n\r\nNo evaluation labels enter fitting or early stopping. Main day6 is followed by\r\nunchanged day5 confirmation only if its conditional gain gate passes.\r\n\"\"\"\r\nimport gc\r\nimport importlib.util\r\nimport json\r\nimport time\r\nfrom pathlib import Path\r\n\r\nimport catboost\r\nimport numpy as np\r\nfrom catboost import CatBoostClassifier, Pool\r\n\r\nPARAMS=dict(iterations=1600,depth=8,learning_rate=.04,l2_leaf_reg=10.,\r\n            loss_function=\"Logloss\",task_type=\"GPU\",devices=\"0\",border_count=63,\r\n            bootstrap_type=\"Bernoulli\",subsample=.8,random_seed=6801,\r\n            boosting_type=\"Plain\",thread_count=4,allow_writing_files=False,\r\n            verbose=100)\r\nBLEND=.25\r\n\r\n\r\ndef helpers(root):\r\n    spec=importlib.util.spec_from_file_location(\"ctr_cb_gpu_helpers\",Path(root)/\"runner.py\")\r\n    module=importlib.util.module_from_spec(spec)\r\n    spec.loader.exec_module(module)\r\n    return module\r\n\r\n\r\ndef predict(model,features):\r\n    out=[]\r\n    for off in range(0,len(features),25000):\r\n        x=features.block(off,min(off+25000,len(features)))\r\n        out.append(model.predict_proba(x,thread_count=4)[:,1])\r\n    return np.concatenate(out)\r\n\r\n\r\ndef training_weights(neural_root, day, labels, h):\r\n    \"\"\"Label-free target history distribution, with exact fitting-row alignment.\"\"\"\r\n    neural_root=Path(neural_root)\r\n    nm=json.loads((neural_root/\"neural_manifest.json\").read_text())\r\n    for name in [\"events.npz\", \"val5.npz\", \"val6.npz\"]:\r\n        assert h.digest(neural_root/name)==nm[\"files\"][name][\"sha256\"]\r\n    with np.load(neural_root/\"events.npz\") as z:\r\n        timestamp=z[\"timestamp\"].copy()\r\n        target=z[\"target\"].copy()\r\n        bins=np.digitize((z[\"hist\"]>=0).sum(1),[0,4,9,19],right=True)\r\n    shares=[]\r\n    for d in [5,6]:\r\n        with np.load(neural_root/f\"val{d}.npz\") as z:\r\n            b=np.repeat(bins[z[\"indices\"]],10)\r\n            w=z[\"weights\"]\r\n            share=np.bincount(b,weights=w,minlength=5)/w.sum()\r\n            shares.append(share)\r\n    assert np.allclose(shares[0],shares[1],atol=1e-7)\r\n    target_share=shares[0]\r\n    event_day=(timestamp-int(nm[\"start\"]))//86400\r\n    selected=(event_day>=day-3)&(event_day<day)\r\n    expected=(target[selected,None]==np.arange(10)[None,:]).astype(\"float32\").ravel()\r\n    assert np.array_equal(expected,labels), \"Fitting labels and event chronology differ\"\r\n    assert (timestamp[selected]<int(nm[\"start\"])+day*86400).all()\r\n    b=np.repeat(bins[selected],10)\r\n    source_share=np.bincount(b,minlength=5)/len(b)\r\n    assert (source_share>0).all()\r\n    weights=(target_share/source_share)[b].astype(\"float32\")\r\n    weights/=weights.mean()\r\n    assert np.isfinite(weights).all() and (weights>0).all()\r\n    return weights,dict(target_history_share=target_share.tolist(),\r\n                        training_history_share=source_share.tolist(),\r\n                        weight_min=float(weights.min()),weight_max=float(weights.max()),\r\n                        alignment=\"exact chronological labels; strictly pre-cutoff events\",\r\n                        population=\"unlabeled evaluation history distribution reconstructed from fixed weights\")\r\n\r\n\r\ndef fit(root,output,manifest,day,h,neural_root):\r\n    started=time.monotonic()\r\n    output.mkdir(parents=True,exist_ok=False)\r\n    days=list(range(day-3,day))\r\n    validation=manifest[\"validation\"][str(day)]\r\n    specs=[manifest[\"train\"][str(d)] for d in days]\r\n    labels=[]\r\n    for spec in specs:\r\n        meta=json.loads((root/spec[\"metadata\"]).read_text())\r\n        assert meta[\"last_timestamp\"]<validation[\"cutoff\"]\r\n        labels.append(np.load(root/spec[\"labels\"]))\r\n    y=np.concatenate(labels).astype(\"float32\")\r\n    weights,weight_recipe=training_weights(neural_root,day,y,h)\r\n    feature=h.FeatureBlocks(root,[s[\"features\"] for s in specs])\r\n    assert len(feature)==len(y)\r\n    recipe=dict(history_importance=weight_recipe,parameters=PARAMS,blend_weight=BLEND,day=day,fit_days=days,\r\n                fit_rows=len(y),columns=354,validation_cutoff=validation[\"cutoff\"],\r\n                input_manifest_sha256=h.digest(root/\"bundle_manifest.json\"),\r\n                source_sha256=h.digest(Path(__file__)),catboost=catboost.__version__,\r\n                selection=\"fixed before scoring; no eval_set or early stopping\")\r\n    (output/\"recipe.json\").write_text(json.dumps(recipe,indent=2))\r\n    temporary=output/\"fit_features.npy\"\r\n    assert temporary.resolve().parent==output.resolve()\r\n    matrix=np.lib.format.open_memmap(temporary,mode=\"w+\",dtype=\"float32\",shape=(len(y),354))\r\n    for off in range(0,len(y),50000):\r\n        end=min(off+50000,len(y));matrix[off:end]=feature.block(off,end)\r\n        if off%500000==0:print(dict(stage=\"prepare_features\",day=day,rows=end,total=len(y)),flush=True)\r\n    matrix.flush()\r\n    pool=Pool(matrix,label=y,weight=weights,thread_count=4)\r\n    model=CatBoostClassifier(**PARAMS)\r\n    print(dict(stage=\"catboost_gpu_fit\",day=day,rows=len(y),iterations=PARAMS[\"iterations\"]),flush=True)\r\n    model.fit(pool)\r\n    model_path=output/\"model.cbm\"\r\n    model.save_model(str(model_path))\r\n    del pool\r\n    matrix._mmap.close()\r\n    del matrix,y,weights\r\n    gc.collect()\r\n    # Remove only the scratch array authored by this fresh experiment.\r\n    assert temporary.resolve().parent==output.resolve()\r\n    temporary.unlink()\r\n    with np.load(root/validation[\"data\"]) as z:\r\n        val={k:z[k].copy() for k in z.files}\r\n    xv=h.FeatureBlocks(root,[validation[\"features\"]])\r\n    assert len(xv)==len(val[\"y\"]) and len(xv)%10==0\r\n    raw=predict(model,xv)\r\n    assert np.isfinite(raw).all() and ((raw>=0)&(raw<=1)).all()\r\n    pred=h.rank((1-BLEND)*h.rank(val[\"baseline\"])+BLEND*h.rank(raw))\r\n    m=h.metrics(val[\"y\"],pred,val[\"baseline\"],val[\"weights\"])\r\n    restored=CatBoostClassifier()\r\n    restored.load_model(str(model_path))\r\n    replay=predict(restored,xv)\r\n    replay_error=float(np.max(np.abs(replay-raw)))\r\n    assert replay_error<1e-12,replay_error\r\n    m.update(day=day,raw_auc=float(h.roc_auc_score(val[\"y\"],raw)),rows=len(raw),\r\n             saved_model_replay_max_error=replay_error,saved_checkpoint_replay=\"PASS\",\r\n             model_sha256=h.digest(model_path),elapsed_seconds=round(time.monotonic()-started,1),\r\n             official_auc=None,csv_ready=False)\r\n    np.savez_compressed(output/\"predictions.npz\",raw=raw,pred=pred,**val)\r\n    (output/\"report.json\").write_text(json.dumps(m,indent=2))\r\n    print(json.dumps(m,indent=2),flush=True)\r\n    return m\r\n\r\n\r\ndef run(root,output,neural_root):\r\n    root,output=Path(root),Path(output)\r\n    h=helpers(root)\r\n    manifest=h.load_bundle(root,verify=True)\r\n    output.mkdir(parents=True,exist_ok=False)\r\n    (output/\"catboost_gpu_weighted_runner.py\").write_text(Path(__file__).read_text())\r\n    reports=[fit(root,output/\"day6\",manifest,6,h,neural_root)]\r\n    if reports[0][\"positive_gate\"]:\r\n        reports.append(fit(root,output/\"day5\",manifest,5,h,neural_root))\r\n    eligible=len(reports)==2 and all(m[\"positive_gate\"] for m in reports)\r\n    summary=dict(reports=reports,eligible_for_review=eligible,target=.83,\r\n                 target_achieved=bool(eligible and reports[0][\"history_adjusted_auc\"]>=.83),\r\n                 csv_ready=False,official_auc=None,\r\n                 note=\"Same reused main dates; fixed history-weighted CatBoost GPU expert. Local metric, not official forecast.\")\r\n    (output/\"summary.json\").write_text(json.dumps(summary,indent=2))\r\n    return summary\r\n",encoding="utf-8")
assert hashlib.sha256(weighting_path.read_bytes()).hexdigest()=="133e47d002c174d6f15c090f2fffc7f0ef63e4b287f3cc6b1d9a30b02e7b2d9c"
source_path=Path("/kaggle/working/tabular_choice_runner.py")
source_path.write_text("\"\"\"Permutation-equivariant neural choice expert over causal numeric features.\n\nFitting and normalization use prior dates only. A fixed blend is scored on\nday6 and confirmed unchanged on day5 only after a positive gain gate.\n\"\"\"\nimport gc\nimport importlib.util\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nRECIPE = dict(epochs=8, batch_size=1024, learning_rate=.0005,\n              weight_decay=.005, dropout=.15, seed=6802, blend_weight=.1,\n              hidden=128, representation=64)\n\n\ndef load_module(name, path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\nclass ChoiceNet(nn.Module):\n    def __init__(self, mean, scale, recipe):\n        super().__init__()\n        self.register_buffer('mean', mean.clone())\n        self.register_buffer('scale', scale.clone())\n        hidden, dim = recipe['hidden'], recipe['representation']\n        self.encoder = nn.Sequential(nn.Linear(len(mean), hidden), nn.ReLU(),\n                                     nn.Linear(hidden, dim), nn.ReLU())\n        self.candidate = nn.Sequential(nn.Linear(3*dim, 64), nn.ReLU(),\n                                       nn.Dropout(recipe['dropout']), nn.Linear(64, 1))\n        self.outside = nn.Sequential(nn.Linear(2*dim, 32), nn.ReLU(), nn.Linear(32, 1))\n        # Sensible initial no-click mass; no dependency on candidate position.\n        nn.init.constant_(self.candidate[-1].bias, -2.)\n        nn.init.zeros_(self.outside[-1].bias)\n\n    def forward(self, x):\n        x = ((x-self.mean)/self.scale).clamp(-10, 10)\n        z = self.encoder(x)\n        avg, peak = z.mean(1), z.max(1).values\n        context = torch.cat([avg, peak], -1)\n        combined = torch.cat([z, context[:, None, :].expand(-1, 10, -1)], -1)\n        return torch.cat([self.candidate(combined).squeeze(-1), self.outside(context)], -1)\n\n\n@torch.no_grad()\ndef predict(model, features, device):\n    model.eval()\n    result = []\n    for off in range(0, len(features), 20000):\n        x = torch.as_tensor(features.block(off, min(off+20000, len(features))), device=device)\n        p = model(x.reshape(-1, 10, 354)).softmax(1)[:, :10]\n        result.append(p.cpu().numpy().ravel())\n    return np.concatenate(result)\n\n\ndef fit(root, neural_root, output, manifest, day, h, weighting, device):\n    started = time.monotonic()\n    output.mkdir(parents=True, exist_ok=False)\n    torch.manual_seed(RECIPE['seed'])\n    specs = [manifest['train'][str(d)] for d in range(day-3, day)]\n    validation = manifest['validation'][str(day)]\n    labels = []\n    for s in specs:\n        meta = json.loads((root/s['metadata']).read_text())\n        assert meta['last_timestamp'] < validation['cutoff']\n        labels.append(np.load(root/s['labels']))\n    y = np.concatenate(labels).astype('float32')\n    assert len(y) % 10 == 0\n    event_y = y.reshape(-1, 10)\n    assert ((event_y == 0) | (event_y == 1)).all() and (event_y.sum(1) <= 1).all()\n    target = np.where(event_y.sum(1) > 0, event_y.argmax(1), 10)\n    weights, weight_recipe = weighting.training_weights(neural_root, day, y, h)\n    weights = weights.reshape(-1, 10)\n    assert np.array_equal(weights, np.repeat(weights[:, :1], 10, axis=1))\n    features = h.FeatureBlocks(root, [s['features'] for s in specs])\n    assert len(features) == len(y)\n    # Transfer one block at a time. No second full host feature matrix.\n    x = torch.empty((len(y), 354), device=device)\n    for off in range(0, len(y), 50000):\n        end = min(off+50000, len(y))\n        x[off:end] = torch.as_tensor(features.block(off, end), device=device)\n    variance, mean = torch.var_mean(x, dim=0, unbiased=False)\n    scale = variance.sqrt().clamp_min(.01)\n    assert torch.isfinite(mean).all() and torch.isfinite(scale).all()\n    x = x.reshape(-1, 10, 354)\n    target = torch.as_tensor(target, device=device, dtype=torch.long)\n    weight = torch.as_tensor(weights[:, 0].copy(), device=device)\n    model = ChoiceNet(mean, scale, RECIPE).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=RECIPE['learning_rate'],\n                                  weight_decay=RECIPE['weight_decay'])\n    recipe = dict(recipe=RECIPE, day=day, fit_days=list(range(day-3, day)),\n                  fit_rows=len(y), validation_cutoff=validation['cutoff'],\n                  weighting=weight_recipe, source_sha256=h.digest(Path(__file__)),\n                  manifest_sha256=h.digest(root/'bundle_manifest.json'),\n                  torch_version=torch.__version__,\n                  selection='Fixed epochs and blend before scoring; no validation optimization')\n    (output/'recipe.json').write_text(json.dumps(recipe, indent=2))\n    for epoch in range(RECIPE['epochs']):\n        model.train()\n        order = torch.randperm(len(x), device=device)\n        total = mass = 0.\n        for off in range(0, len(x), RECIPE['batch_size']):\n            ix = order[off:off+RECIPE['batch_size']]\n            optimizer.zero_grad(set_to_none=True)\n            losses = nn.functional.cross_entropy(model(x[ix]), target[ix], reduction='none')\n            loss = (losses*weight[ix]).sum()/weight[ix].sum()\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), 5.)\n            optimizer.step()\n            total += float((losses.detach()*weight[ix]).sum())\n            mass += float(weight[ix].sum())\n        print(dict(day=day, epoch=epoch+1, epochs=RECIPE['epochs'],\n                   loss=total/mass, elapsed_seconds=round(time.monotonic()-started, 1)), flush=True)\n    checkpoint = output/'model.pt'\n    torch.save(model.state_dict(), checkpoint)\n    del x, target, weight, optimizer\n    gc.collect()\n    if device == 'cuda': torch.cuda.empty_cache()\n    with np.load(root/validation['data']) as z:\n        val = {key:z[key].copy() for key in z.files}\n    xv = h.FeatureBlocks(root, [validation['features']])\n    assert len(xv) == len(val['y'])\n    raw = predict(model, xv, device)\n    assert np.isfinite(raw).all() and ((raw >= 0) & (raw <= 1)).all()\n    assert (raw.reshape(-1, 10).sum(1) <= 1.00001).all()\n    pred = h.rank((1-RECIPE['blend_weight'])*h.rank(val['baseline'])\n                  + RECIPE['blend_weight']*h.rank(raw))\n    m = h.metrics(val['y'], pred, val['baseline'], val['weights'])\n    restored = ChoiceNet(mean, scale, RECIPE).to(device)\n    restored.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))\n    replay = predict(restored, xv, device)\n    error = float(np.max(np.abs(raw-replay)))\n    assert error < 1e-7, error\n    m.update(day=day, raw_auc=float(h.roc_auc_score(val['y'], raw)),\n             saved_model_replay_max_error=error, saved_checkpoint_replay='PASS',\n             model_sha256=h.digest(checkpoint), rows=len(raw),\n             elapsed_seconds=round(time.monotonic()-started, 1), csv_ready=False, official_auc=None)\n    np.savez_compressed(output/'predictions.npz', raw=raw, pred=pred, **val)\n    (output/'report.json').write_text(json.dumps(m, indent=2))\n    print(json.dumps(m, indent=2), flush=True)\n    del model, restored\n    gc.collect()\n    if device == 'cuda': torch.cuda.empty_cache()\n    return m\n\n\ndef run(root, neural_root, output, weighting_path, device='cuda'):\n    root, neural_root, output = Path(root), Path(neural_root), Path(output)\n    assert device == 'cpu' or torch.cuda.is_available()\n    torch.set_num_threads(4)\n    h = load_module('tabular_choice_helpers', root/'runner.py')\n    weighting = load_module('tabular_choice_weights', weighting_path)\n    manifest = h.load_bundle(root, verify=True)\n    output.mkdir(parents=True, exist_ok=False)\n    (output/'tabular_choice_runner.py').write_text(Path(__file__).read_text())\n    (output/'weighting_source.py').write_text(Path(weighting_path).read_text())\n    reports = [fit(root, neural_root, output/'day6', manifest, 6, h, weighting, device)]\n    if reports[0]['positive_gate']:\n        reports.append(fit(root, neural_root, output/'day5', manifest, 5, h, weighting, device))\n    eligible = len(reports) == 2 and all(r['positive_gate'] for r in reports)\n    summary = dict(reports=reports, eligible_for_review=eligible, target=.83,\n                   target_achieved=bool(eligible and reports[0]['history_adjusted_auc'] >= .83),\n                   csv_ready=False, official_auc=None,\n                   note='Reused main validation dates. A local metric, not an official forecast.')\n    (output/'summary.json').write_text(json.dumps(summary, indent=2))\n    return summary\n",encoding="utf-8")
assert hashlib.sha256(source_path.read_bytes()).hexdigest()=="018f3bcd537bf602eec79e7ae7ff04c61f1f62ab6066f1faaace5ed80a8ac187"
spec=importlib.util.spec_from_file_location("ctr_tabular_choice",source_path)
choice=importlib.util.module_from_spec(spec)
spec.loader.exec_module(choice)
RUN_NAME="ctr-tabular-choice-001"
output=Path("/kaggle/working")/RUN_NAME
print("Frozen numeric-choice recipe:",choice.RECIPE)
summary=choice.run(DATA_ROOT,NEURAL_ROOT,output,weighting_path)
summary

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import json, zipfile
print("Experiment:", RUN_NAME)
report = json.loads((output / "summary.json").read_text())
for result in report["reports"]:
    print("Day", result["day"], "Adjusted AUC:", round(result["history_adjusted_auc"], 9), "Ordinary AUC:", round(result["ordinary_auc"], 9), "Gate:", result["positive_gate"])
print("Target 0.83 reached:", report["target_achieved"])
archive = Path("/kaggle/working") / (RUN_NAME + "-results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in output.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(output))
display(FileLink(str(archive)))
